# RedGuard model server on Colab (Ollama + a tunnel)

Serves a 7-10B open model from a free Colab GPU so RedGuard on your laptop can use it instead of a local Ollama.
Guide: `docs/colab_guide.md`. Run the cells top to bottom. **Runtime > Change runtime type > T4 GPU first.**

The tunnel URL is public and has no password: anyone who has it can use your GPU. Do not share it, and stop the
notebook when you are done (last cell).

In [ ]:
# 1. Is there a GPU? You want a Tesla T4 (15 GB) or better.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install Ollama (about 1-2 minutes). zstd, pciutils and lshw let the installer find the GPU.
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# 3. Choose the model. 4-bit builds of 7-10B fit a 15 GB T4 with room for context.
MODEL = "granite3.3:8b"      # same family as the 2B you have used; alternatives are in the guide
CONTEXT = 8192               # tokens of context per request (Lab mode prompts grow with the run)

import os, subprocess, time, requests
os.environ.update({
    "OLLAMA_HOST": "127.0.0.1:11434",     # only reachable from this VM; the tunnel connects locally
    "OLLAMA_KEEP_ALIVE": "24h",           # never unload the model between requests
    "OLLAMA_NUM_PARALLEL": "1",           # one request at a time: reproducible and fits the GPU
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
})
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/version", timeout=1)
        break
    except Exception:
        time.sleep(1)
print("Ollama server:", requests.get("http://127.0.0.1:11434/api/version").json())

In [ ]:
# 4. Download the model (about 5 GB) and load it once, so the first real request is not slow.
!ollama pull {MODEL}
r = requests.post("http://127.0.0.1:11434/api/generate",
                  json={"model": MODEL, "prompt": "Say OK.", "stream": False, "keep_alive": "24h"}, timeout=900)
print("warm-up reply:", r.json()["response"])
!ollama ps

In [ ]:
# 5. Open a tunnel. Cloudflare's quick tunnel needs no account. The URL changes every time you run this cell.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import re, threading
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        URL = found.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep draining so cloudflared never blocks
print("Tunnel URL:", URL)

In [ ]:
# 6. Test through the tunnel exactly as RedGuard will call it (the OpenAI-compatible endpoint under /v1).
t0 = time.time()
r = requests.post(URL + "/v1/chat/completions", timeout=120, json={
    "model": MODEL, "temperature": 0, "seed": 0, "max_tokens": 16,
    "messages": [{"role": "user", "content": "Reply with the single word: ready"}]})
print(r.status_code, r.json()["choices"][0]["message"]["content"], f"({time.time() - t0:.1f}s)")
print()
print("Put these lines in RedGuard's .env, then restart the backend:")
print(f"OLLAMA_BASE_URL={URL}")
print(f"OLLAMA_MODEL={MODEL}")
print(f"LAB_MODEL={MODEL}")
print("OLLAMA_TIMEOUT=120")

In [ ]:
# 7. Look here while a run is going: is the model on the GPU, and is the server healthy?
!ollama ps
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv
!tail -n 5 /content/ollama.log

In [ ]:
# 8. Record what served the results (put this in your pre-registration page).
!ollama --version
print(requests.get("http://127.0.0.1:11434/api/tags").json())

In [ ]:
# 9. Stop everything when you are finished (also: Runtime > Disconnect and delete runtime).
tunnel.terminate(); server.terminate()